# 第 5 章 パーセプトロン

点を直線で 2 クラスに分けます。誤分類した点だけを使ってモデルを動かすパーセプトロントリックを確かめます。

対応する記事: [第 5 章 パーセプトロン（Polyglot Notebook（F#） の言語版）](../../../docs/article/grokking-machine-learning/fsharp/ch05.md)

実装本体: `apps/grokking-ml-fsharp/src/`

## セットアップ

実装本体（`../src/GrokkingMl/`）を `#load` で読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

VS Code の [Polyglot Notebooks 拡張](https://marketplace.visualstudio.com/items?itemName=ms-dotnettools.dotnet-interactive-vscode) で開くか、Jupyter に .NET Interactive カーネルを登録して実行します。

```bash
dotnet tool install -g Microsoft.dotnet-interactive
dotnet interactive jupyter install
jupyter lab notebooks/
```

In [1]:
#load "../src/GrokkingMl/Ch05Perceptron.fs"

open GrokkingMl.Ch05Perceptron

## データセット

原著と同じ「悲しい／楽しい」文の分類データです。特徴量は単語 `aack` と `beep` の出現回数、ラベルは 1 が「楽しい」です。

In [2]:
let points: Point list =
    [ [ 1.0; 0.0 ]; [ 0.0; 2.0 ]; [ 1.0; 1.0 ]; [ 1.0; 2.0 ]
      [ 1.0; 3.0 ]; [ 2.0; 2.0 ]; [ 2.0; 3.0 ]; [ 3.0; 2.0 ] ]

let labels = [ 0; 0; 0; 0; 1; 1; 1; 1 ]

List.zip points labels
|> List.iter (fun (point, label) ->
    printfn "aack=%.0f beep=%.0f → %s" point[0] point[1] (if label = 1 then "楽しい" else "悲しい"))

aack=

1

 beep=

0

 → 

...

3

 beep=

2

 → 

楽しい

## トリックは誤分類した点だけを動かす

**正しく分類できている点では、モデルがまったく変わりません。** これが第 6 章のロジスティック回帰との決定的な違いです。

In [3]:
let model = { Weights = [ 1.0; 2.0 ]; Bias = -4.0 }
printfn "元のモデル       %A" model
printfn "正解した点を渡す %A" (perceptronTrick 0.1 model [ 1.0; 2.0 ] 1)
printfn "誤分類の点を渡す %A" (perceptronTrick 0.1 model [ 1.0; 1.0 ] 1)

元のモデル       

{ Weights = [1.0; 2.0]
  Bias = -4.0 }

正解した点を渡す 

{ Weights = [1.0; 2.0]
  Bias = -4.0 }

誤分類の点を渡す 

{ Weights = [1.1; 2.1]
  Bias = -3.9 }

## 学習

In [4]:
let trained, errors = perceptronAlgorithm 0.01 1000 0 points labels

printfn "重み   %A" (trained.Weights |> List.map (sprintf "%.4f"))
printfn "バイアス %.4f" trained.Bias
printfn "正解率  %.2f" (accuracy trained points labels)

重み   

["0.0400"; "0.0200"]

バイアス 

-0.0800

正解率  

1.00

## パーセプトロン誤差の落とし穴

**初期状態（全パラメータ 0）の誤差は 0 です。** すべての点が境界線上にあるため、誤分類していてもスコアの絶対値が 0 だからです。正解率は 0.5 しかないのに、誤差関数は最良と報告します。

学習の進み具合を見るなら、誤差ではなく **正解率** を見るべきです。

In [5]:
let initial = { Weights = [ 0.0; 0.0 ]; Bias = 0.0 }
printfn "初期の平均誤差 %.4f" (meanPerceptronError initial points labels)
printfn "初期の正解率   %.2f" (accuracy initial points labels)
printfn ""
printfn "学習中の最大誤差 %.4f" (List.max errors)
printfn "最終の平均誤差   %.4f" (List.last errors)
printfn "最終の正解率     %.2f" (accuracy trained points labels)

初期の平均誤差 

0.0000

初期の正解率   

0.50

学習中の最大誤差 

0.0300

最終の平均誤差   

0.0000

最終の正解率     

1.00

## 試してみる

分類では **重みの絶対値に意味がありません**。比率と符号だけが境界線を決めます。すべてを 100 倍しても、予測はまったく同じです。

In [6]:
let scaled =
    { Weights = trained.Weights |> List.map (fun w -> w * 100.0)
      Bias = trained.Bias * 100.0 }

printfn "元のモデル   %A" (points |> List.map (predict trained))
printfn "100 倍モデル %A" (points |> List.map (predict scaled))
printfn "正解         %A" labels

元のモデル   

[0; 0; 0; 0; 1; 1; 1; 1]

100 倍モデル 

[0; 0; 0; 0; 1; 1; 1; 1]

正解         

[0; 0; 0; 0; 1; 1; 1; 1]